# Lift, Splat, Shoot: a standalone walkthrough
This notebook defines the current **practical** segmentation architecture locally. It uses synthetic CPU data, random initialization, and no network or dataset. It is not a reproduced paper result.

`images → camera encoder → depth × context → frustum → ego voxels → BEV decoder → logits`


In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import efficientnet_b0, resnet18, EfficientNet_B0_Weights
torch.manual_seed(0)
torch.set_num_threads(1)


## 1. Camera encoder and lift
Each camera is processed independently. The encoder predicts context `c` and a categorical depth distribution `p(d)`; the lifted feature at depth `d` is `p(d)c`. Shapes: images `[B,N,3,H,W]`, depth `[B,N,D,h,w]`, context `[B,N,C,h,w]`. The stride-16 skip and stride-32 features are fused before prediction.


In [ ]:
class CameraEncoder(nn.Module):
    def __init__(self, depth_bins, context_channels, pretrained=False):
        super().__init__()
        if depth_bins <= 0 or context_channels <= 0: raise ValueError('channels must be positive')
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.features = efficientnet_b0(weights=weights).features[:8]
        self.depth_bins, self.context_channels = depth_bins, context_channels
        self.fuse = nn.Sequential(nn.Conv2d(432, 512, 3, padding=1, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True), nn.Conv2d(512, 512, 3, padding=1, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        self.projection = nn.Conv2d(512, depth_bins + context_channels, 1)
    def forward(self, images):
        if images.ndim != 5 or images.shape[2] != 3: raise ValueError('images must have shape [B,N,3,H,W]')
        batch, cameras, channels, height, width = images.shape
        if height < 32 or width < 32 or height % 32 or width % 32: raise ValueError('image dimensions must be divisible by 32')
        x = images.reshape(batch * cameras, channels, height, width)
        for stage in self.features[:6]: x = stage(x)
        skip = x
        for stage in self.features[6:8]: x = stage(x)
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=True)
        x = self.projection(self.fuse(torch.cat((skip, x), dim=1)))
        depth = x[:, :self.depth_bins].softmax(dim=1)
        context = x[:, self.depth_bins:]
        return depth.reshape(batch, cameras, self.depth_bins, *depth.shape[-2:]), context.reshape(batch, cameras, self.context_channels, *context.shape[-2:])


In [ ]:
camera_probe = CameraEncoder(2, 8).eval()
with torch.no_grad():
    probe_depth, probe_context = camera_probe(torch.zeros(1, 1, 3, 32, 64))
assert probe_depth.shape == (1, 1, 2, 2, 4) and probe_context.shape == (1, 1, 8, 2, 4)
assert torch.allclose(probe_depth.sum(dim=2), torch.ones(1, 1, 2, 4))


## 2. Geometry
Frustum points `(u,v,d)` are sampled in input-image pixels. Undo any image post-transform, then use `K⁻¹[ud,vd,d]` and camera-to-ego extrinsics. Shapes: `[B,N,D,h,w,3]`; coordinates are in metres after unprojection.


In [ ]:
def make_frustum(height, width, depths, device=None, image_size=None):
    if height <= 0 or width <= 0: raise ValueError('frustum dimensions must be positive')
    image_height, image_width = image_size if image_size is not None else (height, width)
    if image_height <= 0 or image_width <= 0: raise ValueError('image dimensions must be positive')
    d = torch.as_tensor(depths, device=device)
    if not d.is_floating_point(): d = d.to(dtype=torch.float32)
    y, x = torch.meshgrid(torch.linspace(0, image_height - 1, height, dtype=d.dtype, device=d.device), torch.linspace(0, image_width - 1, width, dtype=d.dtype, device=d.device), indexing='ij')
    return torch.stack((x[None].expand(len(d), -1, -1), y[None].expand(len(d), -1, -1), d[:, None, None].expand(-1, height, width)), -1)

def camera_to_ego(points, intrinsics, extrinsics, post_rots=None, post_trans=None):
    if points.ndim != 6 or intrinsics.shape[:2] != points.shape[:2] or extrinsics.shape[:2] != points.shape[:2]: raise ValueError('batch and camera axes must match')
    b, c, d, h, w, _ = points.shape
    pix = points.reshape(b, c, -1, 3)
    if (post_rots is None) != (post_trans is None): raise ValueError('post transforms must be provided together')
    if post_rots is not None:
        if post_rots.shape != (b, c, 3, 3) or post_trans.shape != (b, c, 3): raise ValueError('invalid post-transform shape')
        pix = torch.linalg.solve(post_rots, (pix - post_trans.unsqueeze(2)).transpose(-1, -2)).transpose(-1, -2)
    xyz = torch.linalg.solve(intrinsics, torch.cat((pix[..., :2] * pix[..., 2:3], pix[..., 2:3]), -1).transpose(-1, -2)).transpose(-1, -2)
    return ((extrinsics[..., :3, :3].unsqueeze(2) @ xyz.unsqueeze(-1)).squeeze(-1) + extrinsics[..., :3, 3].unsqueeze(2)).reshape(b, c, d, h, w, 3)


In [ ]:
frustum_probe = make_frustum(2, 2, [2.0], image_size=(8, 8))[None, None]
geometry_probe = camera_to_ego(frustum_probe, torch.eye(3)[None, None], torch.eye(4)[None, None])
assert torch.allclose(geometry_probe[0, 0, 0, -1, -1], torch.tensor([14., 14., 2.]))


## 3. Splat and BEV decoder
Sum lifted features in half-open voxel bins. The output is `[B,Z×C,X,Y]`, with height bins concatenated in z-major order. A ResNet-block BEV decoder returns logits on the original BEV grid.


In [ ]:
class VoxelPool(nn.Module):
    def __init__(self, x_bounds, y_bounds, z_bounds):
        super().__init__()
        bounds = (x_bounds, y_bounds, z_bounds); sizes = []
        for lower, upper, step in bounds:
            if not all(math.isfinite(v) for v in (lower, upper, step)) or step <= 0 or upper <= lower: raise ValueError('invalid voxel bounds')
            count = (upper - lower) / step
            if round(count) < 1 or not math.isclose(count, round(count), rel_tol=1e-7, abs_tol=1e-7): raise ValueError('voxel bounds must define an integral number of bins')
            sizes.append(int(round(count)))
        self.grid_size = tuple(sizes)
        self.bounds = tuple(tuple(float(v) for v in bound) for bound in bounds)
    def get_extra_state(self): return {'bounds': self.bounds}
    def set_extra_state(self, state):
        if state.get('bounds') != self.bounds: raise RuntimeError('checkpoint voxel bounds do not match')
    def forward(self, points, features):
        if points.ndim != 6 or points.shape[-1] != 3 or features.ndim != 6: raise ValueError('invalid shape')
        if not points.is_floating_point() or not features.is_floating_point(): raise TypeError('floating point required')
        if points.shape[:-1] != features.shape[:-1] or points.device != features.device: raise ValueError('sample dimensions and device must match')
        batch, channels = points.shape[0], features.shape[-1]; nx, ny, nz = self.grid_size
        lower = points.new_tensor([v[0] for v in self.bounds]); upper = points.new_tensor([v[1] for v in self.bounds]); step = points.new_tensor([v[2] for v in self.bounds])
        valid = ((points >= lower) & (points < upper)).all(dim=-1)
        xyz = torch.floor((points[valid] - lower) / step).long()
        xyz = torch.minimum(xyz.clamp_min(0), torch.tensor((nx-1, ny-1, nz-1), device=points.device))
        batch_index = torch.arange(batch, device=points.device).view(batch, 1, 1, 1, 1).expand(points.shape[:-1])[valid]
        index = ((batch_index * nz + xyz[:, 2]) * nx + xyz[:, 0]) * ny + xyz[:, 1]
        flat = features.new_zeros((batch * nz * nx * ny, channels)).index_add(0, index, features[valid])
        return flat.reshape(batch, nz, nx, ny, channels).permute(0, 1, 4, 2, 3).reshape(batch, nz * channels, nx, ny)

class BEVEncoder(nn.Module):
    def __init__(self, in_channels, out_channels=1):
        super().__init__()
        if in_channels <= 0 or out_channels <= 0: raise ValueError('channels must be positive')
        self.in_channels = in_channels; trunk = resnet18(weights=None, zero_init_residual=True)
        self.stem = nn.Sequential(nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False), trunk.bn1, trunk.relu)
        self.layer1, self.layer2, self.layer3 = trunk.layer1, trunk.layer2, trunk.layer3
        self.fuse = nn.Sequential(nn.Conv2d(320, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.output = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Conv2d(128, out_channels, 1))
    def forward(self, bev):
        if bev.ndim != 4 or bev.shape[1] != self.in_channels: raise ValueError('invalid BEV features')
        height, width = bev.shape[-2:]
        if height < 8 or width < 8 or height % 8 or width % 8: raise ValueError('BEV dimensions must be divisible by 8')
        skip = self.layer1(self.stem(bev)); x = self.layer3(self.layer2(skip))
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=True)
        x = self.fuse(torch.cat((skip, x), dim=1))
        x = F.interpolate(x, size=(height, width), mode='bilinear', align_corners=True)
        return self.output(x)


In [ ]:
pool_probe = VoxelPool((0, 2, 1), (0, 1, 1), (0, 1, 1))
point_probe = torch.tensor([[[[[[0.2, 0.2, 0.2], [0.3, 0.2, 0.2]]]]]])
feature_probe = torch.tensor([[[[[[2.], [3.]]]]]])
assert pool_probe(point_probe, feature_probe)[0, 0, 0, 0] == 5


## 4. Complete segmentation model
The model combines the camera encoder, calibrated projection, sum pooling and BEV decoder. Defaults are source-oriented; the tiny probe below is a synthetic diagnostic configuration. Pretrained weights and checkpoint parity are not assumed.


In [ ]:
class LSSSegmentation(nn.Module):
    def __init__(self, x_bounds=(-50., 50., 0.5), y_bounds=(-50., 50., 0.5), z_bounds=(-10., 10., 20.), depth_bounds=(4., 45., 1.), context_channels=64, out_channels=1, pretrained=False):
        super().__init__()
        start, stop, step = depth_bounds
        if not all(math.isfinite(v) for v in depth_bounds) or start <= 0 or stop <= start or step <= 0: raise ValueError('invalid depth bounds')
        depth_values = torch.arange(start, stop, step, dtype=torch.float32)
        if depth_values.numel() == 0: raise ValueError('empty depth bins')
        self.register_buffer('depth_values', depth_values)
        self.camera_encoder = CameraEncoder(depth_values.numel(), context_channels, pretrained=pretrained)
        self.voxel_pool = VoxelPool(x_bounds, y_bounds, z_bounds)
        self.bev_encoder = BEVEncoder(context_channels * self.voxel_pool.grid_size[2], out_channels)
    def forward(self, images, intrinsics, extrinsics, post_rots=None, post_trans=None):
        depth, context = self.camera_encoder(images)
        frustum = make_frustum(depth.shape[-2], depth.shape[-1], self.depth_values, device=images.device, image_size=images.shape[-2:])
        points = frustum[None, None].expand(images.shape[0], images.shape[1], -1, -1, -1, -1)
        points = camera_to_ego(points, intrinsics, extrinsics, post_rots=post_rots, post_trans=post_trans)
        lifted = depth.unsqueeze(-1) * context.permute(0, 1, 3, 4, 2).unsqueeze(2)
        return self.bev_encoder(self.voxel_pool(points, lifted))


## 5. Synthetic optimization and inference
Train on a tiny occupancy mask using binary cross entropy with logits. Gradients travel through the BEV decoder, pooled features, depth and context predictions. IoU below is a diagnostic computed on the synthetic batch only.


In [ ]:
model = LSSSegmentation(x_bounds=(-8, 8, 1), y_bounds=(-8, 8, 1), z_bounds=(0, 3, 3), depth_bounds=(1, 3, 1), context_channels=8)
images = torch.randn(1, 2, 3, 32, 64)
k = torch.tensor([[10., 0., 32.], [0., 10., 16.], [0., 0., 1.]])[None, None].expand(1, 2, 3, 3).clone()
extrinsics = torch.eye(4)[None, None].expand(1, 2, 4, 4).clone()
target = torch.ones(1, 1, 16, 16)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
before = model.bev_encoder.output[-1].weight.detach().clone()
optimizer.zero_grad()
logits = model(images, k, extrinsics)
loss = F.binary_cross_entropy_with_logits(logits, target)
loss.backward(); optimizer.step()
assert torch.isfinite(loss) and not torch.equal(before, model.bev_encoder.output[-1].weight)
pred = logits.detach() > 0; truth = target.bool()
intersection = (pred & truth).sum().item(); union = (pred | truth).sum().item()
print('synthetic_loss', float(loss.detach()), 'synthetic_iou', intersection / union if union else None)
print('LSS_NOTEBOOK_COMPLETE')
